In [1]:
# 数据处理库
import sys
import json
import logging
import pandas as pd
import numpy as np
from dataset_deduplicate_utils import*
from configs import DatasetDeduplicateConfig

In [2]:
# 数据集基本信息参数传递
property_col = DatasetDeduplicateConfig._property
ref_col = DatasetDeduplicateConfig._reference
state_var_col = DatasetDeduplicateConfig._state_var

# 动态构建 essential_cols
essential_cols = ['c1', 'c2', 'x1', 'x2']
if property_col: essential_cols.append(property_col)
if ref_col: essential_cols.append(ref_col)
if state_var_col: essential_cols.append(state_var_col)

# 定义组分名称列 (使用 SMILES 作为唯一标识)
str_cols = [
    'c1_ismiles',
    'c2_ismiles',
    'c3_ismiles'
]

# 定义组分比例列
num_cols = [
    'x1',
    'x2', 
    'x3'
]

# 中间结果输出目录
interim_output_dir = DatasetDeduplicateConfig.DEDUPLICATE_DIR

# 定义处理完后要删除的辅助列 (例如原始的 c1, c2 名称列，如果不再需要)
drop_columns = ['c1', 'c2', 'c3']

In [3]:
# 1. 读取文件
print("\n正在加载原始数据集...")
original_df = pd.read_excel(DatasetDeduplicateConfig.DATASET_WITH_IDS)
# =========================================================
# 2. 【阶段一】调用 utils 中的体检与重复项打标函数
# =========================================================
print("\n>>> 启动阶段一：识别重复数据 <<<")
intermediate_df = identify_and_export_duplicates(
    original_df=original_df,
    essential_cols=essential_cols,
    str_cols=str_cols,       # 传入组分标识列
    num_cols=num_cols,       # 传入比例列
    output_dir=interim_output_dir,
    ref_col=ref_col,         # 动态参考文献列名
    state_var_cols=state_var_col   
)

# 安全拦截：如果数据在阶段一被全部洗掉了，直接终止
if intermediate_df is None or intermediate_df.empty:
    print("❌ 数据清洗后无有效数据，程序终止。请检查输入数据质量。")
    import sys
    sys.exit(1)


正在加载原始数据集...

>>> 启动阶段一：识别重复数据 <<<
2026-09-14 09:45:39,915 - INFO - 
=== step 1: 关键字段缺失处理 ===
2026-09-14 09:45:39,916 - WARNING - 检测到 1 个包含关键缺失值的记录，正在删除这些记录。
2026-09-14 09:45:39,920 - INFO - 剩余记录数: 2324
2026-09-14 09:45:39,920 - INFO - 
=== step 2: 异常数值审查与拦截 ===
2026-09-14 09:45:39,938 - INFO -  └─ 未发现非法数值
2026-09-14 09:45:39,939 - INFO - 
=== step3: 构建数值去重键 (仅摩尔分数) ===
2026-09-14 09:45:39,953 - INFO - 
=== 识别重复数据 ===
2026-09-14 09:45:39,954 - INFO - 涉及重复的总行数: 1872
2026-09-14 09:45:40,836 - INFO - 重复数据详情已保存至: D:\envs\codes\DES-DataAlchemy\dataset_pretreatment\tmp_data\pi-star_dataset\deduplicate\raw-ds_dd_details.xlsx


In [4]:
result_df = fuse_and_assemble_dataset(
    data_df=intermediate_df,
    property_col=property_col,
    drop_columns=drop_columns,
    ref_col=ref_col,
    agg_method='mean'
)

result_df.to_excel(DatasetDeduplicateConfig.DATASET_DEDUPLICATED, index=False)

2026-09-14 09:45:40,845 - INFO - 
=== 合并重复数据 (当前聚合策略: mean) ===
2026-09-14 09:45:41,035 - INFO - 已将 1872 行重复数据压缩为 478 条唯一记录。
2026-09-14 09:45:41,036 - INFO - 
=== 生成最终数据集 ===
2026-09-14 09:45:41,038 - INFO - 最终数据集构建完成。总行数: 930
